# Task 1.1 — Core Contribution / Architecture
**Paper:** *Training SVMs Without Offset* — Steinwart, Hush & Scovel, JMLR 2011

**Student:** Kush Agarwal | **Roll No.:** 230050 | NST, Rishihood University, Sonipat

## Step-by-Step Method Description

---

**Step 1: Formulate the Offset-Free Dual**

In a standard SVM, the primal problem has both the weight vector **w** and the bias term **b**. When you derive the dual via Lagrange multipliers and take the derivative with respect to **b**, you get the equality constraint Σ αᵢyᵢ = 0. This paper's core idea is to simply not include **b** in the primal at all. Without **b**, the stationarity condition never appears, so the dual has no equality constraint — just individual box constraints 0 ≤ αᵢ ≤ C. The resulting dual is:

maximize W(α) = Σαᵢ − (1/2) αᵀQα

where Qᵢⱼ = yᵢyⱼk(xᵢ, xⱼ), subject to 0 ≤ αᵢ ≤ C for each i.

- **Reference:** Equation (2) and the derivation in Section 1
- **Purpose:** Removes the coupling between dual variables; each αᵢ can now be optimized independently of the others.

---

**Step 2: Initialize Dual Variables (Cold Start — W0)**

Before the first iteration, set α = 0 (all n dual variables to zero) and set the decision value array fv = 0 as well. The paper calls this the W0 initialization strategy (Section 2.2). All α = 0 trivially satisfies 0 ≤ αᵢ ≤ C, so it is always a valid starting point.

- **Reference:** Section 2.2, Initialization — W0 strategy
- **Purpose:** Provides a feasible starting point. The paper also proposes warm-start variants (W2–W6) that reuse solutions from previous C values to speed up hyperparameter search.

---

**Step 3: Compute the Gradient**

At each iteration, compute the partial derivative of W(α) with respect to each αᵢ:

gᵢ = ∂W/∂αᵢ = 1 − yᵢ · f(xᵢ)

where f(xᵢ) = Σⱼ αⱼ yⱼ k(xⱼ, xᵢ) is the current decision value at training point i.

- **Reference:** Equation (3), Section 2
- **Purpose:** gᵢ tells us which direction to move αᵢ to improve the dual objective. Positive gᵢ means the objective increases if we raise αᵢ; negative means we should lower it.

---

**Step 4: Working Set Selection (WSS)**

Choose which variable to update. The paper proposes and compares multiple strategies in Section 4. For WSS 2 (the best 1D strategy according to the paper), compute the KKT violation for each variable:

- If αᵢ < C: violation = max(gᵢ, 0)
- If αᵢ > 0: violation = max(−gᵢ, 0)

Then pick i* = argmax(violation). This is the variable where moving would improve the objective the most.

- **Reference:** Section 4, WSS 2 (maximum violator)
- **Purpose:** Greedy selection converges faster in practice than random selection. The paper compares several WSS strategies empirically in Section 6.

---

**Step 5: Apply the 1D Closed-Form Update**

The key operation. Fix all αⱼ (j ≠ i*) and find the exact maximum of W(α) over αᵢ* alone. This is a 1D constrained quadratic, and the exact closed-form solution is:

αᵢ* ← clip(αᵢ* + gᵢ* / Kᵢ*ᵢ*, 0, C)

where Kᵢᵢ = Qᵢᵢ = k(xᵢ, xᵢ) is the diagonal entry of the kernel matrix, and clip(·, 0, C) projects back into the feasible box.

- **Reference:** Procedure 1, Section 2
- **Purpose:** This is the entire novelty — a single-variable update with no quadratic sub-problem to solve. The step size 1/Kᵢᵢ comes from differentiating the 1D quadratic w.r.t. αᵢ and setting to zero.

---

**Step 6: Update Decision Values Incrementally**

After changing αᵢ* by Δ = αᵢ*_new − αᵢ*_old, every f(xⱼ) changes:

f(xⱼ) += Δ · yᵢ* · k(xᵢ*, xⱼ)   for all j

Using the precomputed kernel matrix, this is just one row-vector multiply: fv += Δ · yᵢ* · K[i*, :], which costs O(n) per iteration instead of O(n²).

- **Reference:** Implied by Procedure 1, Section 2 (incrementally maintained decision values)
- **Purpose:** Makes each iteration O(n) rather than O(n²). This is the main computational savings from precomputing K.

---

**Step 7: Check KKT Stopping Criterion**

After each update, recompute the gradient and re-check KKT violations for all variables. If the maximum violation across all i is below the tolerance ε, stop. The KKT conditions say that at optimality:

- αᵢ = 0 → gᵢ ≤ 0 (can't improve by increasing)
- 0 < αᵢ < C → gᵢ = 0 (interior point, at zero gradient)
- αᵢ = C → gᵢ ≥ 0 (can't improve by decreasing)

- **Reference:** Section 2.1, Equations (6)–(8)
- **Purpose:** Provides a principled stopping rule — not an ad-hoc iteration count. Theorem 1 (Section 5) guarantees this is reached in finite iterations.

---

**Step 8: Produce the Final Classifier**

Once the algorithm converges, the learned α* defines the decision function:

f(x) = Σᵢ αᵢ* · yᵢ · k(xᵢ, x)

Predict class +1 if f(x) > 0, class −1 if f(x) < 0. Note there is no bias term b.

- **Reference:** Equation (1), Section 1
- **Purpose:** Support vectors (training points with αᵢ > 0) are the only points that contribute to the decision. The offset-free decision surface must pass through or near the origin in the feature space.

---

## Final Summary Sentence

This paper solves the computational bottleneck of standard SVM training — the equality constraint that requires updating two variables at a time — by removing the bias term b from the primal, which makes each dual variable independent and allows a simple closed-form 1D coordinate ascent update; the authors claim their offset-free SVM achieves the same classification performance as LIBSVM on standard benchmarks while enabling simpler optimization code and provably finite convergence.